In [1]:
# 1. 导入必要的库
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest, FactorRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from datetime import datetime, timedelta
import rqdatac as rq
from pathlib import Path
from tqdm import tqdm
import json


In [2]:
# ============================================================================
# Cell 2: 配置
# ============================================================================

# 初始化数据服务
datafeed = get_datafeed()           #RqdataDatafeed类  C:\veighna_studio\Lib\site-packages\vnpy_rqdata\rqdata_datafeed.py
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')

数据服务类型: RqdataDatafeed
初始化结果: True


In [2]:
# ============================================================================
# Cell 3: 路径配置
# ===========================================================================

BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'


# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [4]:
# ============================================================================
# Cell 4: 数据下载
# ============================================================================

vt_index_symbol = "866011.VTRI"
rq_index_symbol = "866011.RI"

# 总时间跨度
start = datetime(2013, 1, 1)
end = datetime.now()
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 4, 7)
interval2 = Interval.DAILY




In [5]:
# 4.1 下载成分股列表

# 下载总时间跨度的动态沪深300股票池股票代码
data = rq.index_components(rq_index_symbol, start_date=datetime(2026,4,24), end_date=end)

# 将rq的合约代码转化内vnpy的合约代码
vt_index_components = {}
for dt, rq_symbols in data.items():
    vt_symbols: list = []

    for rq_symbol in rq_symbols:
        vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
        vt_symbols.append(vt_symbol)

    vt_index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....}


# 保存
lab.save_component_data(vt_index_symbol, vt_index_components)



In [5]:
# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)
print(len(component_symbols))

5441


In [6]:
trading_days = rq.get_trading_dates(  # 获取交易日
            start,
            end,
            market="cn"  # 'cn' 代表中国证券市场
        )
trading_days_str = [d.isoformat() for d in trading_days]
with open(lab.trading_days_path, "w+") as f:
    json.dump(trading_days_str, f, indent=2)

In [9]:
# ============================================================================
# Cell 5: 回测参数配置
# ============================================================================
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=15/10000,
        size=1,
        pricetick=0.0001,
    )

In [7]:
quota = rq.user.get_quota()
print(quota)

{'bytes_used': 7351341, 'bytes_limit': 209715200.0, 'remaining_days': 444, 'license_type': 'EDU'}


In [6]:
# ============================================================================
# Cell 6:
# ============================================================================

# 4.2 下载k线
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)

task_symbols = component_symbols  + ["866011.VTRI"]  # 目标股票
n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'开始下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval2, adjust_type='post')
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        print(f'股票{n} 下载完成')
        lab.save_bar_data(bars)
        print(f'保存  ->  {len(bars)} 条K线\n')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 0/5442 [00:00<?, ?it/s]

下载 300626.SZSE ...


  0%|          | 1/5442 [00:02<4:24:41,  2.92s/it]

股票1
  ->  2242 条K线
下载 600168.SSE ...


  0%|          | 3/5442 [00:03<1:12:22,  1.25it/s]

...

股票3389
  ->  2197 条K线
下载 600479.SSE ...


 62%|██████▏   | 3391/5442 [13:25<07:03,  4.84it/s]

股票3390
  ->  3262 条K线
下载 605088.SSE ...
股票3391
  ->  1410 条K线
下载 301355.SZSE ...


 62%|██████▏   | 3391/5442 [13:25<08:07,  4.21it/s]


QuotaExceeded: Quota exceeded

In [8]:
done = [
  "300626.SZSE",
  "600168.SSE",
  "001258.SZSE",
  "600170.SSE",
  "603496.SSE",
  "001216.SZSE",
  "000545.SZSE",
  "002803.SZSE",
  "603330.SSE",
  "300045.SZSE",
  "002173.SZSE",
  "000078.SZSE",
  "300972.SZSE",
  "002466.SZSE",
  "688603.SSE",
  "300677.SZSE",
  "600428.SSE",
  "301032.SZSE",
  "003029.SZSE",
  "603528.SSE",
  "300797.SZSE",
  "301616.SZSE",
  "301429.SZSE",
  "601222.SSE",
  "301237.SZSE",
  "000692.SZSE",
  "301011.SZSE",
  "688365.SSE",
  "300666.SZSE",
  "300168.SZSE",
  "002485.SZSE",
  "603075.SSE",
  "300548.SZSE",
  "600080.SSE",
  "002412.SZSE",
  "300209.SZSE",
  "300719.SZSE",
  "603900.SSE",
  "600389.SSE",
  "688536.SSE",
  "600679.SSE",
  "603767.SSE",
  "603628.SSE",
  "000517.SZSE",
  "601808.SSE",
  "002847.SZSE",
  "688509.SSE",
  "002414.SZSE",
  "601989.SSE",
  "002041.SZSE",
  "002329.SZSE",
  "300401.SZSE",
  "000529.SZSE",
  "002438.SZSE",
  "300489.SZSE",
  "600361.SSE",
  "301188.SZSE",
  "002887.SZSE",
  "000554.SZSE",
  "300185.SZSE",
  "600690.SSE",
  "002511.SZSE",
  "000559.SZSE",
  "002422.SZSE",
  "603332.SSE",
  "300636.SZSE",
  "688265.SSE",
  "001326.SZSE",
  "688416.SSE",
  "688765.SSE",
  "002312.SZSE",
  "688622.SSE",
  "601882.SSE",
  "300698.SZSE",
  "000820.SZSE",
  "688737.SSE",
  "300285.SZSE",
  "600866.SSE",
  "600962.SSE",
  "301413.SZSE",
  "600610.SSE",
  "300615.SZSE",
  "300881.SZSE",
  "600876.SSE",
  "002891.SZSE",
  "688396.SSE",
  "301459.SZSE",
  "688125.SSE",
  "002863.SZSE",
  "000526.SZSE",
  "600370.SSE",
  "002471.SZSE",
  "300462.SZSE",
  "603005.SSE",
  "300788.SZSE",
  "600359.SSE",
  "000985.SZSE",
  "002533.SZSE",
  "688001.SSE",
  "002076.SZSE",
  "600358.SSE",
  "601999.SSE",
  "688017.SSE",
  "688612.SSE",
  "002536.SZSE",
  "002007.SZSE",
  "600897.SSE",
  "000423.SZSE",
  "000778.SZSE",
  "601595.SSE",
  "002895.SZSE",
  "688515.SSE",
  "600030.SSE",
  "688228.SSE",
  "601899.SSE",
  "603280.SSE",
  "300692.SZSE",
  "601288.SSE",
  "300584.SZSE",
  "300347.SZSE",
  "601377.SSE",
  "600824.SSE",
  "301073.SZSE",
  "002262.SZSE",
  "002583.SZSE",
  "600171.SSE",
  "000546.SZSE",
  "688208.SSE",
  "600975.SSE",
  "605255.SSE",
  "002916.SZSE",
  "000963.SZSE",
  "002033.SZSE",
  "600225.SSE",
  "002472.SZSE",
  "001225.SZSE",
  "300945.SZSE",
  "605068.SSE",
  "300917.SZSE",
  "000848.SZSE",
  "301309.SZSE",
  "600758.SSE",
  "002213.SZSE",
  "300172.SZSE",
  "300500.SZSE",
  "600581.SSE",
  "605318.SSE",
  "600566.SSE",
  "601019.SSE",
  "601886.SSE",
  "001316.SZSE",
  "300325.SZSE",
  "688595.SSE",
  "002806.SZSE",
  "002897.SZSE",
  "301172.SZSE",
  "600261.SSE",
  "688503.SSE",
  "300075.SZSE",
  "605011.SSE",
  "601018.SSE",
  "301273.SZSE",
  "688020.SSE",
  "300388.SZSE",
  "301228.SZSE",
  "301026.SZSE",
  "002060.SZSE",
  "002653.SZSE",
  "002083.SZSE",
  "002497.SZSE",
  "002850.SZSE",
  "300559.SZSE",
  "301017.SZSE",
  "600293.SSE",
  "688615.SSE",
  "600490.SSE",
  "300748.SZSE",
  "600320.SSE",
  "002597.SZSE",
  "603369.SSE",
  "300621.SZSE",
  "600272.SSE",
  "002644.SZSE",
  "603360.SSE",
  "001288.SZSE",
  "002682.SZSE",
  "300896.SZSE",
  "603202.SSE",
  "688707.SSE",
  "001896.SZSE",
  "000987.SZSE",
  "603223.SSE",
  "300175.SZSE",
  "000551.SZSE",
  "688395.SSE",
  "000608.SZSE",
  "300920.SZSE",
  "003022.SZSE",
  "601199.SSE",
  "600871.SSE",
  "300894.SZSE",
  "600138.SSE",
  "301178.SZSE",
  "002190.SZSE",
  "603219.SSE",
  "601096.SSE",
  "603186.SSE",
  "601069.SSE",
  "605188.SSE",
  "300556.SZSE",
  "600939.SSE",
  "300996.SZSE",
  "002036.SZSE",
  "600449.SSE",
  "000731.SZSE",
  "688261.SSE",
  "300778.SZSE",
  "600529.SSE",
  "003009.SZSE",
  "603310.SSE",
  "002320.SZSE",
  "300814.SZSE",
  "688777.SSE",
  "601615.SSE",
  "688299.SSE",
  "001330.SZSE",
  "601579.SSE",
  "600373.SSE",
  "002152.SZSE",
  "603216.SSE",
  "300930.SZSE",
  "600688.SSE",
  "301491.SZSE",
  "301591.SZSE",
  "300882.SZSE",
  "002712.SZSE",
  "002147.SZSE",
  "603014.SSE",
  "002671.SZSE",
  "002195.SZSE",
  "300117.SZSE",
  "000795.SZSE",
  "000801.SZSE",
  "002448.SZSE",
  "600774.SSE",
  "002584.SZSE",
  "002401.SZSE",
  "000402.SZSE",
  "002423.SZSE",
  "301563.SZSE",
  "600547.SSE",
  "688053.SSE",
  "301667.SZSE",
  "300018.SZSE",
  "300257.SZSE",
  "603979.SSE",
  "603321.SSE",
  "605136.SSE",
  "300541.SZSE",
  "300651.SZSE",
  "603716.SSE",
  "301183.SZSE",
  "600559.SSE",
  "688528.SSE",
  "688359.SSE",
  "300658.SZSE",
  "688605.SSE",
  "002170.SZSE",
  "002113.SZSE",
  "601628.SSE",
  "603320.SSE",
  "600507.SSE",
  "002265.SZSE",
  "000611.SZSE",
  "600039.SSE",
  "000748.SZSE",
  "605177.SSE",
  "002454.SZSE",
  "300193.SZSE",
  "603159.SSE",
  "601186.SSE",
  "000597.SZSE",
  "688148.SSE",
  "603960.SSE",
  "300949.SZSE",
  "600665.SSE",
  "301330.SZSE",
  "688181.SSE",
  "688697.SSE",
  "000725.SZSE",
  "600712.SSE",
  "301556.SZSE",
  "000060.SZSE",
  "688651.SSE",
  "003028.SZSE",
  "002773.SZSE",
  "300371.SZSE",
  "002942.SZSE",
  "000568.SZSE",
  "000673.SZSE",
  "603220.SSE",
  "600098.SSE",
  "002129.SZSE",
  "002860.SZSE",
  "300947.SZSE",
  "601059.SSE",
  "002431.SZSE",
  "600636.SSE",
  "688270.SSE",
  "300932.SZSE",
  "000582.SZSE",
  "600800.SSE",
  "600580.SSE",
  "300330.SZSE",
  "300716.SZSE",
  "300746.SZSE",
  "002043.SZSE",
  "603055.SSE",
  "000979.SZSE",
  "301458.SZSE",
  "000022.SZSE",
  "600310.SSE",
  "301508.SZSE",
  "300946.SZSE",
  "000830.SZSE",
  "001325.SZSE",
  "000728.SZSE",
  "603538.SSE",
  "603529.SSE",
  "600967.SSE",
  "300841.SZSE",
  "603268.SSE",
  "002188.SZSE",
  "300702.SZSE",
  "300521.SZSE",
  "301390.SZSE",
  "002627.SZSE",
  "300951.SZSE",
  "605117.SSE",
  "300852.SZSE",
  "688167.SSE",
  "000564.SZSE",
  "300573.SZSE",
  "688435.SSE",
  "002336.SZSE",
  "600678.SSE",
  "688097.SSE",
  "300051.SZSE",
  "000040.SZSE",
  "688787.SSE",
  "603119.SSE",
  "688337.SSE",
  "603176.SSE",
  "001696.SZSE",
  "601061.SSE",
  "002667.SZSE",
  "301500.SZSE",
  "300749.SZSE",
  "002981.SZSE",
  "603920.SSE",
  "600780.SSE",
  "300586.SZSE",
  "300138.SZSE",
  "300174.SZSE",
  "603036.SSE",
  "688555.SSE",
  "301395.SZSE",
  "002822.SZSE",
  "603131.SSE",
  "001299.SZSE",
  "301021.SZSE",
  "000607.SZSE",
  "603025.SSE",
  "603999.SSE",
  "002495.SZSE",
  "300427.SZSE",
  "688591.SSE",
  "600363.SSE",
  "688187.SSE",
  "000913.SZSE",
  "300188.SZSE",
  "000927.SZSE",
  "603173.SSE",
  "603296.SSE",
  "301557.SZSE",
  "301180.SZSE",
  "000791.SZSE",
  "601128.SSE",
  "605111.SSE",
  "002530.SZSE",
  "600750.SSE",
  "300502.SZSE",
  "688195.SSE",
  "301123.SZSE",
  "000519.SZSE",
  "688308.SSE",
  "301191.SZSE",
  "688387.SSE",
  "603325.SSE",
  "001380.SZSE",
  "300187.SZSE",
  "688627.SSE",
  "300129.SZSE",
  "689009.SSE",
  "603888.SSE",
  "000043.SZSE",
  "001400.SZSE",
  "300439.SZSE",
  "002283.SZSE",
  "300040.SZSE",
  "688267.SSE",
  "300472.SZSE",
  "688333.SSE",
  "002759.SZSE",
  "000333.SZSE",
  "002416.SZSE",
  "002762.SZSE",
  "002917.SZSE",
  "600057.SSE",
  "002698.SZSE",
  "688779.SSE",
  "603213.SSE",
  "000610.SZSE",
  "603293.SSE",
  "601528.SSE",
  "002388.SZSE",
  "002734.SZSE",
  "603120.SSE",
  "600674.SSE",
  "000969.SZSE",
  "601799.SSE",
  "600208.SSE",
  "300348.SZSE",
  "301111.SZSE",
  "300861.SZSE",
  "600257.SSE",
  "603269.SSE",
  "688071.SSE",
  "000908.SZSE",
  "300265.SZSE",
  "002977.SZSE",
  "603300.SSE",
  "300745.SZSE",
  "301075.SZSE",
  "603106.SSE",
  "301536.SZSE",
  "688098.SSE",
  "002435.SZSE",
  "002906.SZSE",
  "600560.SSE",
  "600680.SSE",
  "300488.SZSE",
  "301399.SZSE",
  "300660.SZSE",
  "002012.SZSE",
  "603020.SSE",
  "300776.SZSE",
  "301099.SZSE",
  "300408.SZSE",
  "600615.SSE",
  "002079.SZSE",
  "688169.SSE",
  "600624.SSE",
  "300400.SZSE",
  "000780.SZSE",
  "002218.SZSE",
  "603324.SSE",
  "603612.SSE",
  "301611.SZSE",
  "001218.SZSE",
  "601512.SSE",
  "300767.SZSE",
  "300888.SZSE",
  "001336.SZSE",
  "603248.SSE",
  "002480.SZSE",
  "600433.SSE",
  "300212.SZSE",
  "000068.SZSE",
  "603693.SSE",
  "600059.SSE",
  "001279.SZSE",
  "603566.SSE",
  "300873.SZSE",
  "301338.SZSE",
  "600151.SSE",
  "603037.SSE",
  "002432.SZSE",
  "000019.SZSE",
  "000409.SZSE",
  "600582.SSE",
  "601086.SSE",
  "688153.SSE",
  "603341.SSE",
  "002352.SZSE",
  "002368.SZSE",
  "600095.SSE",
  "600436.SSE",
  "603511.SSE",
  "603323.SSE",
  "301003.SZSE",
  "000789.SZSE",
  "002797.SZSE",
  "600190.SSE",
  "301105.SZSE",
  "002482.SZSE",
  "002395.SZSE",
  "002885.SZSE",
  "601231.SSE",
  "300139.SZSE",
  "002950.SZSE",
  "603113.SSE",
  "605500.SSE",
  "002449.SZSE",
  "000920.SZSE",
  "601038.SSE",
  "300869.SZSE",
  "688409.SSE",
  "301128.SZSE",
  "300034.SZSE",
  "301102.SZSE",
  "300121.SZSE",
  "605123.SSE",
  "600037.SSE",
  "600748.SSE",
  "301285.SZSE",
  "603677.SSE",
  "301159.SZSE",
  "605077.SSE",
  "688091.SSE",
  "002966.SZSE",
  "002443.SZSE",
  "600070.SSE",
  "300016.SZSE",
  "688268.SSE",
  "002248.SZSE",
  "601717.SSE",
  "300758.SZSE",
  "001391.SZSE",
  "300705.SZSE",
  "002217.SZSE",
  "000565.SZSE",
  "301566.SZSE",
  "605128.SSE",
  "603090.SSE",
  "002300.SZSE",
  "688252.SSE",
  "301312.SZSE",
  "000990.SZSE",
  "002727.SZSE",
  "688185.SSE",
  "002938.SZSE",
  "301126.SZSE",
  "600635.SSE",
  "002902.SZSE",
  "603725.SSE",
  "603107.SSE",
  "603817.SSE",
  "002563.SZSE",
  "600329.SSE",
  "300011.SZSE",
  "301618.SZSE",
  "300308.SZSE",
  "301132.SZSE",
  "002683.SZSE",
  "002110.SZSE",
  "002947.SZSE",
  "300131.SZSE",
  "301083.SZSE",
  "300905.SZSE",
  "301356.SZSE",
  "603738.SSE",
  "000400.SZSE",
  "002687.SZSE",
  "002568.SZSE",
  "601139.SSE",
  "002518.SZSE",
  "002444.SZSE",
  "002724.SZSE",
  "000066.SZSE",
  "002313.SZSE",
  "600794.SSE",
  "603183.SSE",
  "301213.SZSE",
  "300324.SZSE",
  "603739.SSE",
  "600403.SSE",
  "002567.SZSE",
  "301301.SZSE",
  "001270.SZSE",
  "002460.SZSE",
  "002556.SZSE",
  "688328.SSE",
  "300609.SZSE",
  "688539.SSE",
  "002771.SZSE",
  "300825.SZSE",
  "301381.SZSE",
  "300875.SZSE",
  "300143.SZSE",
  "603633.SSE",
  "600175.SSE",
  "603650.SSE",
  "300228.SZSE",
  "002745.SZSE",
  "300338.SZSE",
  "003041.SZSE",
  "000018.SZSE",
  "000606.SZSE",
  "688685.SSE",
  "002334.SZSE",
  "600278.SSE",
  "300044.SZSE",
  "300221.SZSE",
  "000430.SZSE",
  "002679.SZSE",
  "688269.SSE",
  "002094.SZSE",
  "301489.SZSE",
  "000415.SZSE",
  "688072.SSE",
  "300141.SZSE",
  "002651.SZSE",
  "000937.SZSE",
  "603194.SSE",
  "002020.SZSE",
  "000922.SZSE",
  "603049.SSE",
  "600889.SSE",
  "601686.SSE",
  "601900.SSE",
  "600999.SSE",
  "002816.SZSE",
  "000042.SZSE",
  "601921.SSE",
  "600855.SSE",
  "002174.SZSE",
  "600268.SSE",
  "301040.SZSE",
  "688137.SSE",
  "300224.SZSE",
  "301255.SZSE",
  "000892.SZSE",
  "600989.SSE",
  "605055.SSE",
  "000017.SZSE",
  "002006.SZSE",
  "688458.SSE",
  "002622.SZSE",
  "000419.SZSE",
  "300772.SZSE",
  "688305.SSE",
  "688179.SSE",
  "002848.SZSE",
  "002849.SZSE",
  "600187.SSE",
  "000965.SZSE",
  "600243.SSE",
  "688403.SSE",
  "000682.SZSE",
  "002068.SZSE",
  "000566.SZSE",
  "600114.SSE",
  "001296.SZSE",
  "002097.SZSE",
  "603189.SSE",
  "000525.SZSE",
  "300146.SZSE",
  "603205.SSE",
  "300738.SZSE",
  "688472.SSE",
  "300690.SZSE",
  "301033.SZSE",
  "300229.SZSE",
  "300036.SZSE",
  "603217.SSE",
  "002919.SZSE",
  "301683.SZSE",
  "603102.SSE",
  "002856.SZSE",
  "603032.SSE",
  "603506.SSE",
  "603730.SSE",
  "002322.SZSE",
  "000615.SZSE",
  "002971.SZSE",
  "300049.SZSE",
  "301091.SZSE",
  "300074.SZSE",
  "002126.SZSE",
  "301638.SZSE",
  "001234.SZSE",
  "300137.SZSE",
  "002780.SZSE",
  "300314.SZSE",
  "603906.SSE",
  "000632.SZSE",
  "301177.SZSE",
  "600963.SSE",
  "600668.SSE",
  "600887.SSE",
  "603309.SSE",
  "301391.SZSE",
  "301371.SZSE",
  "301499.SZSE",
  "601789.SSE",
  "300913.SZSE",
  "688418.SSE",
  "600005.SSE",
  "600970.SSE",
  "688698.SSE",
  "300176.SZSE",
  "000753.SZSE",
  "600158.SSE",
  "600663.SSE",
  "688238.SSE",
  "300428.SZSE",
  "002084.SZSE",
  "300311.SZSE",
  "300948.SZSE",
  "002837.SZSE",
  "300643.SZSE",
  "603199.SSE",
  "603108.SSE",
  "605016.SSE",
  "688557.SSE",
  "301195.SZSE",
  "688480.SSE",
  "000020.SZSE",
  "000065.SZSE",
  "300837.SZSE",
  "600685.SSE",
  "605116.SSE",
  "601000.SSE",
  "603382.SSE",
  "300454.SZSE",
  "600183.SSE",
  "300058.SZSE",
  "002531.SZSE",
  "002059.SZSE",
  "603717.SSE",
  "002267.SZSE",
  "600229.SSE",
  "600493.SSE",
  "300619.SZSE",
  "300799.SZSE",
  "605090.SSE",
  "002520.SZSE",
  "002616.SZSE",
  "601313.SSE",
  "600146.SSE",
  "000711.SZSE",
  "603236.SSE",
  "600596.SSE",
  "300085.SZSE",
  "600594.SSE",
  "301078.SZSE",
  "002675.SZSE",
  "600255.SSE",
  "600106.SSE",
  "600558.SSE",
  "688347.SSE",
  "002835.SZSE",
  "301028.SZSE",
  "688082.SSE",
  "603123.SSE",
  "002609.SZSE",
  "300681.SZSE",
  "002999.SZSE",
  "301049.SZSE",
  "688311.SSE",
  "000153.SZSE",
  "002706.SZSE",
  "000501.SZSE",
  "603222.SSE",
  "688271.SSE",
  "002086.SZSE",
  "000553.SZSE",
  "002776.SZSE",
  "000669.SZSE",
  "688785.SSE",
  "601868.SSE",
  "688363.SSE",
  "601012.SSE",
  "002907.SZSE",
  "688259.SSE",
  "688391.SSE",
  "002017.SZSE",
  "000758.SZSE",
  "600006.SSE",
  "300482.SZSE",
  "000530.SZSE",
  "002153.SZSE",
  "600578.SSE",
  "002216.SZSE",
  "600880.SSE",
  "002951.SZSE",
  "600567.SSE",
  "002234.SZSE",
  "002163.SZSE",
  "603988.SSE",
  "002202.SZSE",
  "603980.SSE",
  "000679.SZSE",
  "688696.SSE",
  "688459.SSE",
  "002926.SZSE",
  "002612.SZSE",
  "601997.SSE",
  "300282.SZSE",
  "002047.SZSE",
  "603669.SSE",
  "301008.SZSE",
  "301636.SZSE",
  "300411.SZSE",
  "603129.SSE",
  "002424.SZSE",
  "002208.SZSE",
  "600332.SSE",
  "600841.SSE",
  "002647.SZSE",
  "300806.SZSE",
  "688478.SSE",
  "301349.SZSE",
  "688355.SSE",
  "300976.SZSE",
  "002634.SZSE",
  "002355.SZSE",
  "688060.SSE",
  "600343.SSE",
  "688325.SSE",
  "301316.SZSE",
  "688606.SSE",
  "003011.SZSE",
  "002303.SZSE",
  "603937.SSE",
  "000859.SZSE",
  "688796.SSE",
  "600569.SSE",
  "300963.SZSE",
  "603163.SSE",
  "600108.SSE",
  "300476.SZSE",
  "600795.SSE",
  "688015.SSE",
  "301000.SZSE",
  "002927.SZSE",
  "002503.SZSE",
  "000933.SZSE",
  "603893.SSE",
  "301015.SZSE",
  "600732.SSE",
  "300210.SZSE",
  "300202.SZSE",
  "300762.SZSE",
  "600156.SSE",
  "002795.SZSE",
  "002206.SZSE",
  "601311.SSE",
  "002359.SZSE",
  "603398.SSE",
  "688182.SSE",
  "000718.SZSE",
  "002102.SZSE",
  "301552.SZSE",
  "000995.SZSE",
  "688608.SSE",
  "603655.SSE",
  "300277.SZSE",
  "603939.SSE",
  "600755.SSE",
  "002889.SZSE",
  "600707.SSE",
  "688681.SSE",
  "688667.SSE",
  "000977.SZSE",
  "688669.SSE",
  "002635.SZSE",
  "002040.SZSE",
  "603088.SSE",
  "603289.SSE",
  "002402.SZSE",
  "603351.SSE",
  "688313.SSE",
  "600423.SSE",
  "603317.SSE",
  "603755.SSE",
  "688648.SSE",
  "000760.SZSE",
  "600765.SSE",
  "605580.SSE",
  "301361.SZSE",
  "300020.SZSE",
  "600179.SSE",
  "000972.SZSE",
  "600280.SSE",
  "300343.SZSE",
  "300463.SZSE",
  "600869.SSE",
  "300215.SZSE",
  "001382.SZSE",
  "603813.SSE",
  "300980.SZSE",
  "301092.SZSE",
  "002261.SZSE",
  "603985.SSE",
  "300125.SZSE",
  "688506.SSE",
  "688159.SSE",
  "600865.SSE",
  "603608.SSE",
  "688176.SSE",
  "000158.SZSE",
  "603318.SSE",
  "300977.SZSE",
  "603815.SSE",
  "301283.SZSE",
  "688343.SSE",
  "300644.SZSE",
  "600816.SSE",
  "600020.SSE",
  "000828.SZSE",
  "002131.SZSE",
  "002673.SZSE",
  "600744.SSE",
  "300001.SZSE",
  "688231.SSE",
  "600093.SSE",
  "002307.SZSE",
  "688239.SSE",
  "600051.SSE",
  "600270.SSE",
  "301596.SZSE",
  "001389.SZSE",
  "600727.SSE",
  "600576.SSE",
  "002052.SZSE",
  "300918.SZSE",
  "000893.SZSE",
  "601939.SSE",
  "688150.SSE",
  "603863.SSE",
  "688225.SSE",
  "002680.SZSE",
  "002642.SZSE",
  "002714.SZSE",
  "600338.SSE",
  "300355.SZSE",
  "603399.SSE",
  "001358.SZSE",
  "601068.SSE",
  "001209.SZSE",
  "300008.SZSE",
  "300026.SZSE",
  "300991.SZSE",
  "600246.SSE",
  "600551.SSE",
  "300397.SZSE",
  "603421.SSE",
  "601825.SSE",
  "000550.SZSE",
  "002066.SZSE",
  "301037.SZSE",
  "301190.SZSE",
  "300931.SZSE",
  "002787.SZSE",
  "603016.SSE",
  "000717.SZSE",
  "002451.SZSE",
  "600064.SSE",
  "301282.SZSE",
  "001306.SZSE",
  "002965.SZSE",
  "002681.SZSE",
  "300373.SZSE",
  "301507.SZSE",
  "300505.SZSE",
  "600033.SSE",
  "002998.SZSE",
  "600586.SSE",
  "002253.SZSE",
  "002738.SZSE",
  "600864.SSE",
  "300301.SZSE",
  "688556.SSE",
  "300475.SZSE",
  "600115.SSE",
  "600822.SSE",
  "600830.SSE",
  "603166.SSE",
  "600525.SSE",
  "000816.SZSE",
  "000785.SZSE",
  "688529.SSE",
  "301560.SZSE",
  "600273.SSE",
  "000601.SZSE",
  "688376.SSE",
  "603968.SSE",
  "688728.SSE",
  "002824.SZSE",
  "688162.SSE",
  "000836.SZSE",
  "002719.SZSE",
  "300189.SZSE",
  "000901.SZSE",
  "603328.SSE",
  "003025.SZSE",
  "688469.SSE",
  "001979.SZSE",
  "002648.SZSE",
  "002065.SZSE",
  "001219.SZSE",
  "688321.SSE",
  "003015.SZSE",
  "605069.SSE",
  "002367.SZSE",
  "600598.SSE",
  "601665.SSE",
  "300194.SZSE",
  "002654.SZSE",
  "601169.SSE",
  "000802.SZSE",
  "603093.SSE",
  "300295.SZSE",
  "688552.SSE",
  "603086.SSE",
  "002309.SZSE",
  "603466.SSE",
  "600807.SSE",
  "002230.SZSE",
  "300975.SZSE",
  "603385.SSE",
  "000027.SZSE",
  "002883.SZSE",
  "688663.SSE",
  "600647.SSE",
  "002958.SZSE",
  "301153.SZSE",
  "301149.SZSE",
  "601399.SSE",
  "301498.SZSE",
  "300807.SZSE",
  "002151.SZSE",
  "688573.SSE",
  "601100.SSE",
  "002923.SZSE",
  "603040.SSE",
  "001289.SZSE",
  "002555.SZSE",
  "688596.SSE",
  "300372.SZSE",
  "300235.SZSE",
  "603311.SSE",
  "002252.SZSE",
  "601616.SSE",
  "601567.SSE",
  "001221.SZSE",
  "002114.SZSE",
  "688059.SSE",
  "301258.SZSE",
  "300558.SZSE",
  "300763.SZSE",
  "300684.SZSE",
  "605007.SSE",
  "301307.SZSE",
  "600870.SSE",
  "301118.SZSE",
  "600252.SSE",
  "605186.SSE",
  "603601.SSE",
  "300479.SZSE",
  "603823.SSE",
  "688212.SSE",
  "301599.SZSE",
  "300664.SZSE",
  "688543.SSE",
  "300542.SZSE",
  "605033.SSE",
  "603198.SSE",
  "300183.SZSE",
  "688621.SSE",
  "301166.SZSE",
  "002366.SZSE",
  "300418.SZSE",
  "002591.SZSE",
  "600211.SSE",
  "600616.SSE",
  "300108.SZSE",
  "688329.SSE",
  "002630.SZSE",
  "300254.SZSE",
  "600099.SSE",
  "300812.SZSE",
  "600069.SSE",
  "605169.SSE",
  "688323.SSE",
  "600630.SSE",
  "000803.SZSE",
  "688625.SSE",
  "603299.SSE",
  "688289.SSE",
  "301055.SZSE",
  "000782.SZSE",
  "002774.SZSE",
  "603077.SSE",
  "600461.SSE",
  "000688.SZSE",
  "600277.SSE",
  "000821.SZSE",
  "603122.SSE",
  "600135.SSE",
  "301269.SZSE",
  "002311.SZSE",
  "002831.SZSE",
  "688232.SSE",
  "601118.SSE",
  "603826.SSE",
  "300575.SZSE",
  "001212.SZSE",
  "600128.SSE",
  "002166.SZSE",
  "000596.SZSE",
  "603227.SSE",
  "002162.SZSE",
  "600439.SSE",
  "603221.SSE",
  "002979.SZSE",
  "600782.SSE",
  "603898.SSE",
  "000050.SZSE",
  "300906.SZSE",
  "600642.SSE",
  "688129.SSE",
  "600105.SSE",
  "301513.SZSE",
  "300217.SZSE",
  "000572.SZSE",
  "002866.SZSE",
  "603533.SSE",
  "002119.SZSE",
  "603010.SSE",
  "301050.SZSE",
  "603969.SSE",
  "301487.SZSE",
  "300780.SZSE",
  "300590.SZSE",
  "603358.SSE",
  "603758.SSE",
  "000012.SZSE",
  "300267.SZSE",
  "688123.SSE",
  "601568.SSE",
  "000905.SZSE",
  "300904.SZSE",
  "300458.SZSE",
  "300161.SZSE",
  "600335.SSE",
  "688192.SSE",
  "000033.SZSE",
  "600792.SSE",
  "000028.SZSE",
  "002193.SZSE",
  "003816.SZSE",
  "301205.SZSE",
  "601778.SSE",
  "600595.SSE",
  "603866.SSE",
  "300195.SZSE",
  "688108.SSE",
  "300676.SZSE",
  "603170.SSE",
  "000426.SZSE",
  "001255.SZSE",
  "300849.SZSE",
  "002507.SZSE",
  "603916.SSE",
  "603666.SSE",
  "002165.SZSE",
  "002142.SZSE",
  "002109.SZSE",
  "600746.SSE",
  "300095.SZSE",
  "300808.SZSE",
  "002548.SZSE",
  "600222.SSE",
  "002621.SZSE",
  "002430.SZSE",
  "300473.SZSE",
  "002623.SZSE",
  "603729.SSE",
  "603098.SSE",
  "600682.SSE",
  "001257.SZSE",
  "002857.SZSE",
  "688258.SSE",
  "600418.SSE",
  "688216.SSE",
  "000416.SZSE",
  "301045.SZSE",
  "300557.SZSE",
  "600330.SSE",
  "002143.SZSE",
  "603598.SSE",
  "688616.SSE",
  "300263.SZSE",
  "002356.SZSE",
  "002532.SZSE",
  "600997.SSE",
  "603085.SSE",
  "000993.SZSE",
  "002948.SZSE",
  "002445.SZSE",
  "002660.SZSE",
  "002638.SZSE",
  "300145.SZSE",
  "600406.SSE",
  "601727.SSE",
  "600289.SSE",
  "301696.SZSE",
  "300517.SZSE",
  "688175.SSE",
  "600163.SSE",
  "003000.SZSE",
  "002526.SZSE",
  "300063.SZSE",
  "001317.SZSE",
  "603639.SSE",
  "002373.SZSE",
  "300403.SZSE",
  "688781.SSE",
  "603992.SSE",
  "002728.SZSE",
  "300916.SZSE",
  "603212.SSE",
  "300530.SZSE",
  "002049.SZSE",
  "000010.SZSE",
  "300935.SZSE",
  "300962.SZSE",
  "688717.SSE",
  "600637.SSE",
  "002016.SZSE",
  "603392.SSE",
  "301559.SZSE",
  "603838.SSE",
  "301152.SZSE",
  "603030.SSE",
  "002812.SZSE",
  "001238.SZSE",
  "600498.SSE",
  "688410.SSE",
  "600763.SSE",
  "688136.SSE",
  "603558.SSE",
  "000998.SZSE",
  "002901.SZSE",
  "603390.SSE",
  "300902.SZSE",
  "300340.SZSE",
  "600601.SSE",
  "688601.SSE",
  "300148.SZSE",
  "300433.SZSE",
  "688729.SSE",
  "000792.SZSE",
  "600238.SSE",
  "300569.SZSE",
  "601010.SSE",
  "301119.SZSE",
  "300101.SZSE",
  "600177.SSE",
  "605399.SSE",
  "301290.SZSE",
  "688549.SSE",
  "000038.SZSE",
  "600377.SSE",
  "301468.SZSE",
  "600801.SSE",
  "301192.SZSE",
  "300192.SZSE",
  "002778.SZSE",
  "605003.SSE",
  "603927.SSE",
  "688030.SSE",
  "000861.SZSE",
  "600593.SSE",
  "600048.SSE",
  "688112.SSE",
  "600120.SSE",
  "002237.SZSE",
  "601929.SSE",
  "301656.SZSE",
  "300030.SZSE",
  "300234.SZSE",
  "603379.SSE",
  "600331.SSE",
  "600836.SSE",
  "600692.SSE",
  "002200.SZSE",
  "688373.SSE",
  "002702.SZSE",
  "600325.SSE",
  "688301.SSE",
  "601188.SSE",
  "300289.SZSE",
  "688716.SSE",
  "002839.SZSE",
  "603489.SSE",
  "300903.SZSE",
  "300999.SZSE",
  "301408.SZSE",
  "300665.SZSE",
  "300990.SZSE",
  "600806.SSE",
  "600835.SSE",
  "000159.SZSE",
  "002328.SZSE",
  "688535.SSE",
  "603890.SSE",
  "603335.SSE",
  "688767.SSE",
  "688029.SSE",
  "600354.SSE",
  "301396.SZSE",
  "002756.SZSE",
  "601238.SSE",
  "000636.SZSE",
  "603168.SSE",
  "001230.SZSE",
  "300064.SZSE",
  "688609.SSE",
  "002785.SZSE",
  "600185.SSE",
  "300545.SZSE",
  "601975.SSE",
  "000766.SZSE",
  "603709.SSE",
  "001332.SZSE",
  "000807.SZSE",
  "301060.SZSE",
  "002123.SZSE",
  "002859.SZSE",
  "301626.SZSE",
  "002840.SZSE",
  "000524.SZSE",
  "603021.SSE",
  "300992.SZSE",
  "002207.SZSE",
  "301010.SZSE",
  "000862.SZSE",
  "688197.SSE",
  "300491.SZSE",
  "300997.SZSE",
  "688408.SSE",
  "000902.SZSE",
  "600118.SSE",
  "600909.SSE",
  "600230.SSE",
  "000852.SZSE",
  "001333.SZSE",
  "688008.SSE",
  "000968.SZSE",
  "300572.SZSE",
  "300856.SZSE",
  "603949.SSE",
  "600537.SSE",
  "688652.SSE",
  "002429.SZSE",
  "603929.SSE",
  "300617.SZSE",
  "300798.SZSE",
  "002677.SZSE",
  "600328.SSE",
  "688575.SSE",
  "600540.SSE",
  "300668.SZSE",
  "688633.SSE",
  "002209.SZSE",
  "301260.SZSE",
  "301587.SZSE",
  "002189.SZSE",
  "603003.SSE",
  "688219.SSE",
  "603073.SSE",
  "600857.SSE",
  "002061.SZSE",
  "002363.SZSE",
  "002280.SZSE",
  "002843.SZSE",
  "688065.SSE",
  "603773.SSE",
  "688468.SSE",
  "600353.SSE",
  "300109.SZSE",
  "601857.SSE",
  "688443.SSE",
  "601101.SSE",
  "300178.SZSE",
  "688006.SSE",
  "603665.SSE",
  "002519.SZSE",
  "688739.SSE",
  "001314.SZSE",
  "688775.SSE",
  "600608.SSE",
  "301287.SZSE",
  "600395.SSE",
  "000919.SZSE",
  "300050.SZSE",
  "600200.SSE",
  "600008.SSE",
  "688708.SSE",
  "300817.SZSE",
  "600787.SSE",
  "688227.SSE",
  "002124.SZSE",
  "300770.SZSE",
  "002297.SZSE",
  "301629.SZSE",
  "600300.SSE",
  "300520.SZSE",
  "002957.SZSE",
  "300675.SZSE",
  "002758.SZSE",
  "002204.SZSE",
  "603275.SSE",
  "601689.SSE",
  "301308.SZSE",
  "002960.SZSE",
  "002400.SZSE",
  "600769.SSE",
  "603818.SSE",
  "300611.SZSE",
  "002737.SZSE",
  "603406.SSE",
  "002748.SZSE",
  "002760.SZSE",
  "300497.SZSE",
  "603060.SSE",
  "300471.SZSE",
  "003040.SZSE",
  "600365.SSE",
  "002557.SZSE",
  "300216.SZSE",
  "003004.SZSE",
  "300259.SZSE",
  "002976.SZSE",
  "002605.SZSE",
  "300376.SZSE",
  "603056.SSE",
  "000039.SZSE",
  "300518.SZSE",
  "600346.SSE",
  "000959.SZSE",
  "002655.SZSE",
  "601811.SSE",
  "688388.SSE",
  "300731.SZSE",
  "688586.SSE",
  "000422.SZSE",
  "600713.SSE",
  "000818.SZSE",
  "300241.SZSE",
  "000823.SZSE",
  "603787.SSE",
  "603409.SSE",
  "688665.SSE",
  "002149.SZSE",
  "300391.SZSE",
  "601678.SSE",
  "002566.SZSE",
  "603203.SSE",
  "300724.SZSE",
  "301518.SZSE",
  "603013.SSE",
  "301337.SZSE",
  "300332.SZSE",
  "001222.SZSE",
  "600808.SSE",
  "601718.SSE",
  "601600.SSE",
  "600109.SSE",
  "600182.SSE",
  "000633.SZSE",
  "600321.SSE",
  "603777.SSE",
  "002549.SZSE",
  "002425.SZSE",
  "002457.SZSE",
  "300055.SZSE",
  "603590.SSE",
  "300591.SZSE",
  "002761.SZSE",
  "000014.SZSE",
  "603809.SSE",
  "002867.SZSE",
  "688588.SSE",
  "002517.SZSE",
  "300649.SZSE",
  "300926.SZSE",
  "600741.SSE",
  "688253.SSE",
  "301052.SZSE",
  "301230.SZSE",
  "002270.SZSE",
  "603659.SSE",
  "603171.SSE",
  "600422.SSE",
  "600221.SSE",
  "002399.SZSE",
  "688522.SSE",
  "601598.SSE",
  "300499.SZSE",
  "300173.SZSE",
  "300086.SZSE",
  "300238.SZSE",
  "300519.SZSE",
  "002590.SZSE",
  "301598.SZSE",
  "301086.SZSE",
  "300782.SZSE",
  "688202.SSE",
  "002182.SZSE",
  "688577.SSE",
  "301365.SZSE",
  "600753.SSE",
  "600928.SSE",
  "603262.SSE",
  "002139.SZSE",
  "301515.SZSE",
  "300891.SZSE",
  "600415.SSE",
  "603656.SSE",
  "603091.SSE",
  "603063.SSE",
  "603069.SSE",
  "688319.SSE",
  "000523.SZSE",
  "603810.SSE",
  "688351.SSE",
  "601827.SSE",
  "300196.SZSE",
  "600391.SSE",
  "300170.SZSE",
  "688026.SSE",
  "002087.SZSE",
  "000506.SZSE",
  "002344.SZSE",
  "300214.SZSE",
  "002410.SZSE",
  "600336.SSE",
  "002559.SZSE",
  "600392.SSE",
  "603352.SSE",
  "603516.SSE",
  "600101.SSE",
  "301179.SZSE",
  "000089.SZSE",
  "002050.SZSE",
  "603345.SSE",
  "603190.SSE",
  "605199.SSE",
  "301210.SZSE",
  "603207.SSE",
  "600491.SSE",
  "002274.SZSE",
  "001356.SZSE",
  "603990.SSE",
  "300915.SZSE",
  "300669.SZSE",
  "002544.SZSE",
  "688193.SSE",
  "605122.SSE",
  "002015.SZSE",
  "603267.SSE",
  "600327.SSE",
  "600817.SSE",
  "001226.SZSE",
  "000779.SZSE",
  "300362.SZSE",
  "000045.SZSE",
  "301531.SZSE",
  "600157.SSE",
  "002288.SZSE",
  "001378.SZSE",
  "300179.SZSE",
  "002140.SZSE",
  "000850.SZSE",
  "688326.SSE",
  "600757.SSE",
  "688358.SSE",
  "000150.SZSE",
  "000798.SZSE",
  "600425.SSE",
  "000810.SZSE",
  "603843.SSE",
  "300094.SZSE",
  "002324.SZSE",
  "002463.SZSE",
  "300773.SZSE",
  "603306.SSE",
  "000776.SZSE",
  "002306.SZSE",
  "002092.SZSE",
  "002676.SZSE",
  "688381.SSE",
  "603083.SSE",
  "000971.SZSE",
  "300425.SZSE",
  "300043.SZSE",
  "600295.SSE",
  "600770.SSE",
  "002827.SZSE",
  "000726.SZSE",
  "600408.SSE",
  "603602.SSE",
  "603721.SSE",
  "688789.SSE",
  "601005.SSE",
  "301310.SZSE",
  "603095.SSE",
  "300750.SZSE",
  "002250.SZSE",
  "002421.SZSE",
  "002224.SZSE",
  "603158.SSE",
  "600873.SSE",
  "603028.SSE",
  "600819.SSE",
  "688031.SSE",
  "300801.SZSE",
  "000683.SZSE",
  "600754.SSE",
  "603130.SSE",
  "001308.SZSE",
  "688589.SSE",
  "301133.SZSE",
  "603615.SSE",
  "605577.SSE",
  "000707.SZSE",
  "300886.SZSE",
  "603836.SSE",
  "300895.SZSE",
  "300855.SZSE",
  "002205.SZSE",
  "300546.SZSE",
  "301234.SZSE",
  "300567.SZSE",
  "002484.SZSE",
  "002931.SZSE",
  "301668.SZSE",
  "300232.SZSE",
  "688315.SSE",
  "301362.SZSE",
  "600516.SSE",
  "002793.SZSE",
  "002838.SZSE",
  "605319.SSE",
  "301041.SZSE",
  "000593.SZSE",
  "002273.SZSE",
  "603828.SSE",
  "300088.SZSE",
  "300539.SZSE",
  "603600.SSE",
  "301065.SZSE",
  "300219.SZSE",
  "300382.SZSE",
  "605268.SSE",
  "002332.SZSE",
  "603786.SSE",
  "600210.SSE",
  "603776.SSE",
  "300789.SZSE",
  "600249.SSE",
  "300160.SZSE",
  "688559.SSE",
  "603089.SSE",
  "002528.SZSE",
  "603407.SSE",
  "300073.SZSE",
  "688576.SSE",
  "301069.SZSE",
  "301377.SZSE",
  "301522.SZSE",
  "002343.SZSE",
  "688349.SSE",
  "301206.SZSE",
  "600728.SSE",
  "000626.SZSE",
  "300009.SZSE",
  "600019.SSE",
  "603418.SSE",
  "300938.SZSE",
  "600771.SSE",
  "000032.SZSE",
  "600247.SSE",
  "300936.SZSE",
  "300323.SZSE",
  "300336.SZSE",
  "603882.SSE",
  "600166.SSE",
  "301117.SZSE",
  "688222.SSE",
  "300673.SZSE",
  "002796.SZSE",
  "002370.SZSE",
  "300223.SZSE",
  "002171.SZSE",
  "300911.SZSE",
  "002289.SZSE",
  "688079.SSE",
  "002201.SZSE",
  "601898.SSE",
  "600926.SSE",
  "688790.SSE",
  "002172.SZSE",
  "300618.SZSE",
  "601456.SSE",
  "300528.SZSE",
  "000856.SZSE",
  "002487.SZSE",
  "600028.SSE",
  "300080.SZSE",
  "300623.SZSE",
  "300632.SZSE",
  "002150.SZSE",
  "301529.SZSE",
  "001220.SZSE",
  "600446.SSE",
  "002953.SZSE",
  "688809.SSE",
  "002989.SZSE",
  "601366.SSE",
  "301297.SZSE",
  "002589.SZSE",
  "001379.SZSE",
  "688668.SSE",
  "300147.SZSE",
  "002920.SZSE",
  "603919.SSE",
  "600096.SSE",
  "000511.SZSE",
  "301169.SZSE",
  "000552.SZSE",
  "600207.SSE",
  "600667.SSE",
  "600812.SSE",
  "600271.SSE",
  "001360.SZSE",
  "301516.SZSE",
  "601500.SSE",
  "688592.SSE",
  "601113.SSE",
  "002229.SZSE",
  "301360.SZSE",
  "300152.SZSE",
  "003001.SZSE",
  "603928.SSE",
  "300119.SZSE",
  "301288.SZSE",
  "001201.SZSE",
  "601566.SSE",
  "688210.SSE",
  "300318.SZSE",
  "003033.SZSE",
  "600161.SSE",
  "688582.SSE",
  "300087.SZSE",
  "600838.SSE",
  "688610.SSE",
  "002376.SZSE",
  "002387.SZSE",
  "688287.SSE",
  "002984.SZSE",
  "300543.SZSE",
  "600643.SSE",
  "300029.SZSE",
  "300607.SZSE",
  "600073.SSE",
  "001229.SZSE",
  "300493.SZSE",
  "600309.SSE",
  "000752.SZSE",
  "002069.SZSE",
  "300697.SZSE",
  "600469.SSE",
  "300093.SZSE",
  "002404.SZSE",
  "002339.SZSE",
  "600023.SSE",
  "603517.SSE",
  "601995.SSE",
  "600661.SSE",
  "603766.SSE",
  "600022.SSE",
  "600768.SSE",
  "300721.SZSE",
  "300072.SZSE",
  "600568.SSE",
  "001313.SZSE",
  "300431.SZSE",
  "002739.SZSE",
  "688429.SSE",
  "002450.SZSE",
  "603286.SSE",
  "603353.SSE",
  "605567.SSE",
  "600535.SSE",
  "301383.SZSE",
  "601901.SSE",
  "600038.SSE",
  "301081.SZSE",
  "688255.SSE",
  "688209.SSE",
  "688805.SSE",
  "000096.SZSE",
  "300998.SZSE",
  "601998.SSE",
  "301112.SZSE",
  "603045.SSE",
  "603357.SSE",
  "603661.SSE",
  "301127.SZSE",
  "002767.SZSE",
  "600600.SSE",
  "002775.SZSE",
  "300713.SZSE",
  "002258.SZSE",
  "600883.SSE",
  "601588.SSE",
  "603811.SSE",
  "301221.SZSE",
  "603261.SSE",
  "300594.SZSE",
  "603530.SSE",
  "600067.SSE",
  "002662.SZSE",
  "000720.SZSE",
  "300010.SZSE",
  "688151.SSE",
  "603298.SSE",
  "600376.SSE",
  "603315.SSE",
  "002105.SZSE",
  "688816.SSE",
  "688115.SSE",
  "600130.SSE",
  "300613.SZSE",
  "601229.SSE",
  "300851.SZSE",
  "300452.SZSE",
  "300456.SZSE",
  "002085.SZSE",
  "603259.SSE",
  "603151.SSE",
  "300674.SZSE",
  "688655.SSE",
  "603991.SSE",
  "688233.SSE",
  "002246.SZSE",
  "300389.SZSE",
  "300331.SZSE",
  "000505.SZSE",
  "600121.SSE",
  "600853.SSE",
  "301051.SZSE",
  "300113.SZSE",
  "600648.SSE",
  "301076.SZSE",
  "000048.SZSE",
  "300860.SZSE",
  "000630.SZSE",
  "002656.SZSE",
  "603727.SSE",
  "301512.SZSE",
  "600122.SSE",
  "603917.SSE",
  "300655.SZSE",
  "603583.SSE",
  "603456.SSE",
  "002125.SZSE",
  "300656.SZSE",
  "000638.SZSE",
  "603869.SSE",
  "300107.SZSE",
  "002302.SZSE",
  "002604.SZSE",
  "300328.SZSE",
  "603182.SSE",
  "000069.SZSE",
  "002491.SZSE",
  "688178.SSE",
  "601872.SSE",
  "600828.SSE",
  "600074.SSE",
  "603897.SSE",
  "601163.SSE",
  "000625.SZSE",
  "002290.SZSE",
  "300909.SZSE",
  "301246.SZSE",
  "000929.SZSE",
  "600724.SSE",
  "001280.SZSE",
  "688593.SSE",
  "600435.SSE",
  "002735.SZSE",
  "002641.SZSE",
  "688366.SSE",
  "002333.SZSE",
  "603989.SSE",
  "300820.SZSE",
  "688262.SSE",
  "002881.SZSE",
  "601065.SSE",
  "002562.SZSE",
  "600905.SSE",
  "605162.SSE",
  "000656.SZSE",
  "603879.SSE",
  "688086.SSE",
  "301106.SZSE",
  "002552.SZSE",
  "688256.SSE",
  "601011.SSE",
  "600260.SSE",
  "002198.SZSE",
  "688561.SSE",
  "603933.SSE",
  "300589.SZSE",
  "600603.SSE",
  "688073.SSE",
  "600675.SSE",
  "688067.SSE",
  "603662.SSE",
  "002377.SZSE",
  "300098.SZSE",
  "000046.SZSE",
  "603042.SSE",
  "000680.SZSE",
  "002688.SZSE",
  "301314.SZSE",
  "300507.SZSE",
  "002028.SZSE",
  "600097.SSE",
  "603381.SSE",
  "301004.SZSE",
  "000537.SZSE",
  "688690.SSE",
  "002820.SZSE",
  "600011.SSE",
  "688211.SSE",
  "300297.SZSE",
  "002350.SZSE",
  "600790.SSE",
  "603260.SSE",
  "000488.SZSE",
  "300765.SZSE",
  "600751.SSE",
  "603518.SSE",
  "600149.SSE",
  "002637.SZSE",
  "600571.SSE",
  "000826.SZSE",
  "300885.SZSE",
  "600081.SSE",
  "000425.SZSE",
  "600452.SSE",
  "002570.SZSE",
  "600644.SSE",
  "600496.SSE",
  "688618.SSE",
  "002659.SZSE",
  "605566.SSE",
  "003043.SZSE",
  "688692.SSE",
  "603105.SSE",
  "301311.SZSE",
  "601777.SSE",
  "300627.SZSE",
  "300341.SZSE",
  "688327.SSE",
  "603333.SSE",
  "600927.SSE",
  "300067.SZSE",
  "688800.SSE",
  "000631.SZSE",
  "688283.SSE",
  "605299.SSE",
  "002928.SZSE",
  "605218.SSE",
  "301116.SZSE",
  "603878.SSE",
  "603271.SSE",
  "300134.SZSE",
  "000156.SZSE",
  "603955.SSE",
  "300442.SZSE",
  "600488.SSE",
  "300699.SZSE",
  "000029.SZSE",
  "600382.SSE",
  "300540.SZSE",
  "603135.SSE",
  "688631.SSE",
  "603798.SSE",
  "688121.SSE",
  "002168.SZSE",
  "600139.SSE",
  "300889.SZSE",
  "002269.SZSE",
  "301295.SZSE",
  "603508.SSE",
  "301251.SZSE",
  "603499.SSE",
  "688293.SSE",
  "301321.SZSE",
  "688018.SSE",
  "000915.SZSE",
  "600798.SSE",
  "600193.SSE",
  "002973.SZSE",
  "605298.SSE",
  "603004.SSE",
  "600152.SSE",
  "300937.SZSE",
  "600990.SSE",
  "603657.SSE",
  "300833.SZSE",
  "603402.SSE",
  "301272.SZSE",
  "301333.SZSE",
  "002990.SZSE",
  "688120.SSE",
  "300960.SZSE",
  "605183.SSE",
  "688191.SSE",
  "605208.SSE",
  "688693.SSE",
  "601677.SSE",
  "600689.SSE",
  "002657.SZSE",
  "600340.SSE",
  "603889.SSE",
  "002564.SZSE",
  "301160.SZSE",
  "603370.SSE",
  "603997.SSE",
  "300757.SZSE",
  "000612.SZSE",
  "688083.SSE",
  "688199.SSE",
  "002872.SZSE",
  "688702.SSE",
  "300683.SZSE",
  "605398.SSE",
  "301046.SZSE",
  "002183.SZSE",
  "600973.SSE",
  "301388.SZSE",
  "002955.SZSE",
  "300560.SZSE",
  "300038.SZSE",
  "000762.SZSE",
  "600677.SSE",
  "301002.SZSE",
  "301222.SZSE",
  "301335.SZSE",
  "600592.SSE",
  "301575.SZSE",
  "001339.SZSE",
  "600611.SSE",
  "301059.SZSE",
  "600401.SSE",
  "600463.SSE",
  "688198.SSE",
  "301038.SZSE",
  "002141.SZSE",
  "002039.SZSE",
  "301601.SZSE",
  "300378.SZSE",
  "600322.SSE",
  "688005.SSE",
  "000687.SZSE",
  "600756.SSE",
  "002148.SZSE",
  "301229.SZSE",
  "300973.SZSE",
  "600162.SSE",
  "000021.SZSE",
  "600145.SSE",
  "600000.SSE",
  "301439.SZSE",
  "603138.SSE",
  "300981.SZSE",
  "301025.SZSE",
  "300126.SZSE",
  "000088.SZSE",
  "300062.SZSE",
  "000822.SZSE",
  "603051.SSE",
  "000002.SZSE",
  "600789.SSE",
  "605080.SSE",
  "603637.SSE",
  "301358.SZSE",
  "300857.SZSE",
  "001309.SZSE",
  "600877.SSE",
  "600779.SSE",
  "002560.SZSE",
  "600654.SSE",
  "002225.SZSE",
  "000715.SZSE",
  "600299.SSE",
  "002042.SZSE",
  "600409.SSE",
  "603118.SSE",
  "000911.SZSE",
  "605287.SSE",
  "000004.SZSE",
  "300249.SZSE",
  "600172.SSE",
  "300296.SZSE",
  "600388.SSE",
  "301526.SZSE",
  "002203.SZSE",
  "300069.SZSE",
  "600925.SSE",
  "300565.SZSE",
  "300969.SZSE",
  "600223.SSE",
  "600811.SSE",
  "603339.SSE",
  "000544.SZSE",
  "300602.SZSE",
  "002362.SZSE",
  "300580.SZSE",
  "301057.SZSE",
  "600884.SSE",
  "301291.SZSE",
  "000413.SZSE",
  "300017.SZSE",
  "603329.SSE",
  "000650.SZSE",
  "603948.SSE",
  "601916.SSE",
  "300583.SZSE",
  "000016.SZSE",
  "603567.SSE",
  "601992.SSE",
  "300736.SZSE",
  "300103.SZSE",
  "300862.SZSE",
  "002096.SZSE",
  "688217.SSE",
  "605050.SSE",
  "600575.SSE",
  "300268.SZSE",
  "002220.SZSE",
  "000584.SZSE",
  "688700.SSE",
  "301630.SZSE",
  "603230.SSE",
  "600485.SSE",
  "300480.SZSE",
  "000637.SZSE",
  "300688.SZSE",
  "301592.SZSE",
  "300696.SZSE",
  "002001.SZSE",
  "688526.SSE",
  "000860.SZSE",
  "003019.SZSE",
  "603165.SSE",
  "603883.SSE",
  "688229.SSE",
  "000786.SZSE",
  "600235.SSE",
  "600708.SSE",
  "002219.SZSE",
  "601996.SSE",
  "300978.SZSE",
  "301219.SZSE",
  "301012.SZSE",
  "301053.SZSE",
  "300555.SZSE",
  "002995.SZSE",
  "600891.SSE",
  "603366.SSE",
  "600657.SSE",
  "002228.SZSE",
  "603638.SSE",
  "301503.SZSE",
  "002038.SZSE",
  "603160.SSE",
  "003007.SZSE",
  "000930.SZSE",
  "300853.SZSE",
  "601577.SSE",
  "301317.SZSE",
  "688116.SSE",
  "301327.SZSE",
  "000639.SZSE",
  "002179.SZSE",
  "000982.SZSE",
  "688486.SSE",
  "300370.SZSE",
  "002815.SZSE",
  "002510.SZSE",
  "601609.SSE",
  "000420.SZSE",
  "688531.SSE",
  "300740.SZSE",
  "605337.SSE",
  "002122.SZSE",
  "603308.SSE",
  "688033.SSE",
  "002058.SZSE",
  "601963.SSE",
  "601116.SSE",
  "002750.SZSE",
  "002892.SZSE",
  "603855.SSE",
  "000099.SZSE",
  "002742.SZSE",
  "603187.SSE",
  "300815.SZSE",
  "301080.SZSE",
  "002538.SZSE",
  "301198.SZSE",
  "002314.SZSE",
  "300366.SZSE",
  "600502.SSE",
  "603079.SSE",
  "600036.SSE",
  "000812.SZSE",
  "600508.SSE",
  "600722.SSE",
  "601020.SSE",
  "301121.SZSE",
  "600503.SSE",
  "600186.SSE",
  "600851.SSE",
  "301225.SZSE",
  "001285.SZSE",
  "688722.SSE",
  "300549.SZSE",
  "300610.SZSE",
  "688113.SSE",
  "600483.SSE",
  "300835.SZSE",
  "001260.SZSE",
  "000532.SZSE",
  "600696.SSE",
  "301088.SZSE",
  "002034.SZSE",
  "603193.SSE",
  "600111.SSE",
  "002672.SZSE",
  "601200.SSE",
  "000739.SZSE",
  "002628.SZSE",
  "301520.SZSE",
  "300495.SZSE",
  "603444.SSE",
  "688611.SSE",
  "603400.SSE",
  "600981.SSE",
  "688011.SSE",
  "002617.SZSE",
  "603918.SSE",
  "000722.SZSE",
  "600010.SSE",
  "000806.SZSE",
  "300068.SZSE",
  "002192.SZSE",
  "300240.SZSE",
  "600107.SSE",
  "300377.SZSE",
  "002585.SZSE",
  "000157.SZSE",
  "300245.SZSE",
  "300357.SZSE",
  "600546.SSE",
  "300133.SZSE",
  "300890.SZSE",
  "001390.SZSE",
  "688157.SSE",
  "688420.SSE",
  "600820.SSE",
  "301029.SZSE",
  "002310.SZSE",
  "600369.SSE",
  "002186.SZSE",
  "600562.SSE",
  "300847.SZSE",
  "300819.SZSE",
  "688236.SSE",
  "600867.SSE",
  "688038.SSE",
  "000790.SZSE",
  "300122.SZSE",
  "601688.SSE",
  "600810.SSE",
  "000898.SZSE",
  "601066.SSE",
  "002620.SZSE",
  "600797.SSE",
  "002099.SZSE",
  "002380.SZSE",
  "002643.SZSE",
  "002030.SZSE",
  "300423.SZSE",
  "002877.SZSE",
  "300312.SZSE",
  "301137.SZSE",
  "000767.SZSE",
  "688523.SSE",
  "300387.SZSE",
  "603277.SSE",
  "600088.SSE",
  "301171.SZSE",
  "300120.SZSE",
  "301418.SZSE",
  "600123.SSE",
  "600206.SSE",
  "300317.SZSE",
  "002082.SZSE",
  "002694.SZSE",
  "000916.SZSE",
  "300399.SZSE",
  "002823.SZSE",
  "603589.SSE",
  "002116.SZSE",
  "600918.SSE",
  "001287.SZSE",
  "603231.SSE",
  "000976.SZSE",
  "603983.SSE",
  "000543.SZSE",
  "002271.SZSE",
  "301488.SZSE",
  "688426.SSE",
  "002440.SZSE",
  "002379.SZSE",
  "601117.SSE",
  "300414.SZSE",
  "300291.SZSE",
  "688220.SSE",
  "002080.SZSE",
  "688049.SSE",
  "000528.SZSE",
  "301175.SZSE",
  "300663.SZSE",
  "002799.SZSE",
  "603150.SSE",
  "603065.SSE",
  "300706.SZSE",
  "600378.SSE",
  "002599.SZSE",
  "601015.SSE",
  "603515.SSE",
  "600982.SSE",
  "301528.SZSE",
  "603568.SSE",
  "601339.SSE",
  "000869.SZSE",
  "688100.SSE",
  "002420.SZSE",
  "300007.SZSE",
  "000935.SZSE",
  "601933.SSE",
  "002321.SZSE",
  "000966.SZSE",
  "002481.SZSE",
  "002508.SZSE",
  "001328.SZSE",
  "600583.SSE",
  "300691.SZSE",
  "002301.SZSE",
  "300326.SZSE",
  "601966.SSE",
  "603350.SSE",
  "688818.SSE",
  "600396.SSE",
  "300225.SZSE",
  "688138.SSE",
  "688811.SSE",
  "688032.SSE",
  "300076.SZSE",
  "300645.SZSE",
  "000686.SZSE",
  "301115.SZSE",
  "688114.SSE",
  "301682.SZSE",
  "603276.SSE",
  "688165.SSE",
  "301226.SZSE",
  "603899.SSE",
  "601619.SSE",
  "603376.SSE",
  "688302.SSE",
  "300031.SZSE",
  "601608.SSE",
  "301550.SZSE",
  "002127.SZSE",
  "601599.SSE",
  "002145.SZSE",
  "603565.SSE",
  "600739.SSE",
  "600416.SSE",
  "001324.SZSE",
  "000978.SZSE",
  "000709.SZSE",
  "603185.SSE",
  "000009.SZSE",
  "603017.SSE",
  "688530.SSE",
  "300563.SZSE",
  "000008.SZSE",
  "002832.SZSE",
  "603976.SSE",
  "002319.SZSE",
  "300822.SZSE",
  "605089.SSE",
  "300982.SZSE",
  "301281.SZSE",
  "688248.SSE",
  "002932.SZSE",
  "001267.SZSE",
  "300925.SZSE",
  "600671.SSE",
  "688186.SSE",
  "603218.SSE",
  "688081.SSE",
  "002685.SZSE",
  "002577.SZSE",
  "000581.SZSE",
  "300703.SZSE",
  "301511.SZSE",
  "002645.SZSE",
  "688500.SSE",
  "688330.SSE",
  "600426.SSE",
  "603125.SSE",
  "002842.SZSE",
  "002112.SZSE",
  "002397.SZSE",
  "600236.SSE",
  "301302.SZSE",
  "002035.SZSE",
  "601187.SSE",
  "600521.SSE",
  "001298.SZSE",
  "688278.SSE",
  "300910.SZSE",
  "600063.SSE",
  "300430.SZSE",
  "300252.SZSE",
  "600209.SSE",
  "001301.SZSE",
  "600455.SSE",
  "688450.SSE",
  "002411.SZSE",
  "000712.SZSE",
  "300276.SZSE",
  "600719.SSE",
  "301030.SZSE",
  "301609.SZSE",
  "000697.SZSE",
  "603062.SSE",
  "301096.SZSE",
  "002242.SZSE",
  "301165.SZSE",
  "002700.SZSE",
  "301558.SZSE",
  "300302.SZSE",
  "300106.SZSE",
  "603901.SSE",
  "301231.SZSE",
  "601319.SSE",
  "688676.SSE",
  "600720.SSE",
  "688802.SSE",
  "600259.SSE",
  "688056.SSE",
  "603076.SSE",
  "301380.SZSE",
  "300151.SZSE",
  "300484.SZSE",
  "000886.SZSE",
  "000713.SZSE",
  "002805.SZSE",
  "600201.SSE",
  "605099.SSE",
  "300828.SZSE",
  "600971.SSE",
  "605118.SSE",
  "301027.SZSE",
  "688658.SSE",
  "601801.SSE",
  "002490.SZSE",
  "600082.SSE",
  "300571.SZSE",
  "300071.SZSE",
  "300854.SZSE",
  "600941.SSE",
  "300751.SZSE",
  "688131.SSE",
  "603986.SSE",
  "300956.SZSE",
  "300300.SZSE",
  "601601.SSE",
  "000666.SZSE",
  "000737.SZSE",
  "688711.SSE",
  "000617.SZSE",
  "002592.SZSE",
  "600532.SSE",
  "603688.SSE",
  "002390.SZSE",
  "001231.SZSE",
  "301369.SZSE",
  "300253.SZSE",
  "600137.SSE",
  "002956.SZSE",
  "001323.SZSE",
  "601111.SSE",
  "603856.SSE",
  "688370.SSE",
  "300294.SZSE",
  "001278.SZSE",
  "688281.SSE",
  "002144.SZSE",
  "600793.SSE",
  "603127.SSE",
  "300246.SZSE",
  "300529.SZSE",
  "301501.SZSE",
  "600872.SSE",
  "002327.SZSE",
  "002408.SZSE",
  "300933.SZSE",
  "603050.SSE",
  "600035.SSE",
  "300426.SZSE",
  "000883.SZSE",
  "600545.SSE",
  "002705.SZSE",
  "600988.SSE",
  "605499.SSE",
  "688068.SSE",
  "000585.SZSE",
  "603806.SSE",
  "000948.SZSE",
  "001283.SZSE",
  "300288.SZSE",
  "000677.SZSE",
  "000516.SZSE",
  "600527.SSE",
  "301215.SZSE",
  "688190.SSE",
  "003006.SZSE",
  "300006.SZSE",
  "002936.SZSE",
  "300566.SZSE",
  "300700.SZSE",
  "603305.SSE",
  "603380.SSE",
  "300477.SZSE",
  "600619.SSE",
  "300661.SZSE",
  "600743.SSE",
  "003032.SZSE",
  "300162.SZSE",
  "601798.SSE",
  "000663.SZSE",
  "601860.SSE",
  "002861.SZSE",
  "002576.SZSE",
  "688375.SSE",
  "603367.SSE",
  "300298.SZSE",
  "603950.SSE",
  "300667.SZSE",
  "688223.SSE",
  "300342.SZSE",
  "002330.SZSE",
  "301378.SZSE",
  "603613.SSE",
  "688050.SSE",
  "600032.SSE",
  "300057.SZSE",
  "301007.SZSE",
  "301613.SZSE",
  "600198.SSE",
  "000011.SZSE",
  "300984.SZSE",
  "300264.SZSE",
  "002337.SZSE",
  "688699.SSE",
  "300961.SZSE",
  "300883.SZSE",
  "600716.SSE",
  "000809.SZSE",
  "301398.SZSE",
  "600584.SSE",
  "600749.SSE",
  "001376.SZSE",
  "002690.SZSE",
  "300446.SZSE",
  "002326.SZSE",
  "301603.SZSE",
  "600844.SSE",
  "001368.SZSE",
  "300159.SZSE",
  "688103.SSE",
  "002286.SZSE",
  "688619.SSE",
  "688076.SSE",
  "002175.SZSE",
  "603139.SSE",
  "300786.SZSE",
  "002164.SZSE",
  "603001.SSE",
  "688061.SSE",
  "002708.SZSE",
  "600796.SSE",
  "600459.SSE",
  "601375.SSE",
  "002504.SZSE",
  "000903.SZSE",
  "000909.SZSE",
  "688126.SSE",
  "600617.SSE",
  "002678.SZSE",
  "600506.SSE",
  "002029.SZSE",
  "600908.SSE",
  "002441.SZSE",
  "688069.SSE",
  "688296.SSE",
  "603926.SSE",
  "603297.SSE",
  "300379.SZSE",
  "300287.SZSE",
  "302132.SZSE",
  "000401.SZSE",
  "600004.SSE",
  "300955.SZSE",
  "601107.SSE",
  "601990.SSE",
  "301212.SZSE",
  "301449.SZSE",
  "002992.SZSE",
  "002864.SZSE",
  "688338.SSE",
  "300720.SZSE",
  "600561.SSE",
  "002023.SZSE",
  "601155.SSE",
  "601089.SSE",
  "002462.SZSE",
  "002930.SZSE",
  "605588.SSE",
  "600155.SSE",
  "300352.SZSE",
  "002718.SZSE",
  "003002.SZSE",
  "688602.SSE",
  "301551.SZSE",
  "300859.SZSE",
  "603197.SSE",
  "688550.SSE",
  "300525.SZSE",
  "300353.SZSE",
  "002755.SZSE",
  "600094.SSE",
  "601022.SSE",
  "603285.SSE",
  "300364.SZSE",
  "000951.SZSE",
  "000671.SZSE",
  "300266.SZSE",
  "002024.SZSE",
  "001395.SZSE",
  "000757.SZSE",
  "300227.SZSE",
  "000155.SZSE",
  "600827.SSE",
  "603667.SSE",
  "300250.SZSE",
  "000831.SZSE",
  "605598.SSE",
  "300070.SZSE",
  "002055.SZSE",
  "300385.SZSE",
  "002602.SZSE",
  "002478.SZSE",
  "600703.SSE",
  "601607.SSE",
  "600968.SSE",
  "300775.SZSE",
  "300803.SZSE",
  "603344.SSE",
  "605151.SSE",
  "688143.SSE",
  "600606.SSE",
  "000521.SZSE",
  "600077.SSE",
  "688080.SSE",
  "002829.SZSE",
  "300293.SZSE",
  "600345.SSE",
  "600557.SSE",
  "688401.SSE",
  "002579.SZSE",
  "603459.SSE",
  "603759.SSE",
  "300079.SZSE",
  "001311.SZSE",
  "300437.SZSE",
  "603800.SSE",
  "300562.SZSE",
  "601636.SSE",
  "002474.SZSE",
  "603172.SSE",
  "600305.SSE",
  "301389.SZSE",
  "300424.SZSE",
  "301090.SZSE",
  "002090.SZSE",
  "300848.SZSE",
  "301517.SZSE",
  "002117.SZSE",
  "300870.SZSE",
  "300843.SZSE",
  "002982.SZSE",
  "301602.SZSE",
  "688189.SSE",
  "600579.SSE",
  "300236.SZSE",
  "300672.SZSE",
  "688309.SSE",
  "688551.SSE",
  "601318.SSE",
  "600933.SSE",
  "603117.SSE",
  "002912.SZSE",
  "603707.SSE",
  "603200.SSE",
  "000727.SZSE",
  "301665.SZSE",
  "600076.SSE",
  "688628.SSE",
  "601958.SSE",
  "603697.SSE",
  "002893.SZSE",
  "000877.SZSE",
  "003013.SZSE",
  "002212.SZSE",
  "002539.SZSE",
  "002709.SZSE",
  "603728.SSE",
  "300641.SZSE",
  "600290.SSE",
  "301122.SZSE",
  "002918.SZSE",
  "603803.SSE",
  "600649.SSE",
  "001366.SZSE",
  "002459.SZSE",
  "002715.SZSE",
  "603970.SSE",
  "301107.SZSE",
  "002308.SZSE",
  "688372.SSE",
  "000410.SZSE",
  "603689.SSE",
  "688235.SSE",
  "301093.SZSE",
  "688161.SSE",
  "301167.SZSE",
  "603180.SSE",
  "600977.SSE",
  "002275.SZSE",
  "600285.SSE",
  "300112.SZSE",
  "002027.SZSE",
  "601969.SSE",
  "300200.SZSE",
  "000514.SZSE",
  "300839.SZSE",
  "600291.SSE",
  "000960.SZSE",
  "601008.SSE",
  "300952.SZSE",
  "688291.SSE",
  "002428.SZSE",
  "002817.SZSE",
  "300838.SZSE",
  "301129.SZSE",
  "002810.SZSE",
  "002331.SZSE",
  "002304.SZSE",
  "301131.SZSE",
  "603733.SSE",
  "688590.SSE",
  "001337.SZSE",
  "600802.SSE",
  "002652.SZSE",
  "002391.SZSE",
  "688456.SSE",
  "000750.SZSE",
  "600938.SSE",
  "603061.SSE",
  "300829.SZSE",
  "603363.SSE",
  "002266.SZSE",
  "002697.SZSE",
  "300100.SZSE",
  "300508.SZSE",
  "600614.SSE",
  "688362.SSE",
  "600717.SSE",
  "300460.SZSE",
  "603737.SSE",
  "601330.SSE",
  "600352.SSE",
  "688584.SSE",
  "603687.SSE",
  "002789.SZSE",
  "600767.SSE",
  "603867.SSE",
  "688027.SSE",
  "301139.SZSE",
  "002798.SZSE",
  "000925.SZSE",
  "688367.SSE",
  "300127.SZSE",
  "600066.SSE",
  "002574.SZSE",
  "300821.SZSE",
  "002800.SZSE",
  "300334.SZSE",
  "002691.SZSE",
  "002855.SZSE",
  "688568.SSE",
  "600623.SSE",
  "300116.SZSE",
  "002729.SZSE",
  "002749.SZSE",
  "300114.SZSE",
  "301548.SZSE",
  "002940.SZSE",
  "000887.SZSE",
  "600733.SSE",
  "603209.SSE",
  "002725.SZSE",
  "601336.SSE",
  "300081.SZSE",
  "600745.SSE",
  "000548.SZSE",
  "600965.SSE",
  "000561.SZSE",
  "300653.SZSE",
  "002013.SZSE",
  "301087.SZSE",
  "605286.SSE",
  "002573.SZSE",
  "002614.SZSE",
  "600298.SSE",
  "000555.SZSE",
  "002505.SZSE",
  "600113.SSE",
  "002515.SZSE",
  "688691.SSE",
  "002078.SZSE",
  "002633.SZSE",
  "688617.SSE",
  "002437.SZSE",
  "300792.SZSE",
  "002996.SZSE",
  "301158.SZSE",
  "600612.SSE",
  "301589.SZSE",
  "600269.SSE",
  "688277.SSE",
  "000598.SZSE",
  "600197.SSE",
  "300522.SZSE",
  "600053.SSE",
  "605198.SSE",
  "603000.SSE",
  "688545.SSE",
  "300021.SZSE",
  "688122.SSE",
  "301173.SZSE",
  "600110.SSE",
  "603007.SSE",
  "600764.SSE",
  "300531.SZSE",
  "603312.SSE",
  "002747.SZSE",
  "600501.SSE",
  "000708.SZSE",
  "000049.SZSE",
  "002913.SZSE",
  "301632.SZSE",
  "600355.SSE",
  "301185.SZSE",
  "688683.SSE",
  "603273.SSE",
  "600985.SSE",
  "301580.SZSE",
  "603887.SSE",
  "002259.SZSE",
  "688399.SSE",
  "603278.SSE",
  "300633.SZSE",
  "300614.SZSE",
  "601156.SSE",
  "300747.SZSE",
  "601519.SSE",
  "300638.SZSE",
  "301519.SZSE",
  "002464.SZSE",
  "601669.SSE",
  "688331.SSE",
  "000667.SZSE",
  "002442.SZSE",
  "300612.SZSE",
  "002315.SZSE",
  "688679.SSE",
  "001211.SZSE",
  "300440.SZSE",
  "300604.SZSE",
  "688002.SSE",
  "605336.SSE",
  "300805.SZSE",
  "300115.SZSE",
  "688657.SSE",
  "300710.SZSE",
  "002026.SZSE",
  "002933.SZSE",
  "601696.SSE",
  "002521.SZSE",
  "301043.SZSE",
  "000502.SZSE",
  "002821.SZSE",
  "605377.SSE",
  "603895.SSE",
  "300879.SZSE",
  "002345.SZSE",
  "600522.SSE",
  "000635.SZSE",
  "002021.SZSE",
  "300464.SZSE",
  "600935.SSE",
  "688639.SSE",
  "300205.SZSE",
  "000691.SZSE",
  "002121.SZSE",
  "002985.SZSE",
  "600846.SSE",
  "300536.SZSE",
  "301109.SZSE",
  "600730.SSE",
  "002100.SZSE",
  "003023.SZSE",
  "603987.SSE",
  "300468.SZSE",
  "003012.SZSE",
  "603115.SSE",
  "603963.SSE",
  "002279.SZSE",
  "300729.SZSE",
  "002752.SZSE",
  "001203.SZSE",
  "688721.SSE",
  "002545.SZSE",
  "300033.SZSE",
  "002523.SZSE",
  "603439.SSE",
  "300380.SZSE",
  "002571.SZSE",
  "300726.SZSE",
  "301035.SZSE",
  "000034.SZSE",
  "000678.SZSE",
  "600383.SSE",
  "603520.SSE",
  "301196.SZSE",
  "300158.SZSE",
  "600531.SSE",
  "688807.SSE",
  "688215.SSE",
  "300363.SZSE",
  "002968.SZSE",
  "688310.SSE",
  "301042.SZSE",
  "002498.SZSE",
  "300585.SZSE",
  "600085.SSE",
  "002427.SZSE",
  "601326.SSE",
  "001386.SZSE",
  "603611.SSE",
  "688630.SSE",
  "002868.SZSE",
  "601828.SSE",
  "300281.SZSE",
  "000062.SZSE",
  "002723.SZSE",
  "002608.SZSE",
  "000026.SZSE",
  "600497.SSE",
  "600153.SSE",
  "301608.SZSE",
  "688035.SSE",
  "002871.SZSE",
  "688636.SSE",
  "000006.SZSE",
  "301386.SZSE",
  "605167.SSE",
  "002104.SZSE",
  "000835.SZSE",
  "002910.SZSE",
  "688659.SSE",
  "002858.SZSE",
  "301248.SZSE",
  "603155.SSE",
  "601088.SSE",
  "300694.SZSE",
  "301387.SZSE",
  "600302.SSE",
  "002284.SZSE",
  "000592.SZSE",
  "605365.SSE",
  "300830.SZSE",
  "300124.SZSE",
  "600839.SSE",
  "600318.SSE",
  "688772.SSE",
  "688795.SSE",
  "300004.SZSE",
  "603080.SSE",
  "002900.SZSE",
  "600898.SSE",
  "688201.SSE",
  "002558.SZSE",
  "002624.SZSE",
  "301161.SZSE",
  "603958.SSE",
  "603757.SSE",
  "301658.SZSE",
  "600960.SSE",
  "600016.SSE",
  "601006.SSE",
  "301024.SZSE",
  "688455.SSE",
  "600686.SSE",
  "002535.SZSE",
  "002254.SZSE",
  "603015.SSE",
  "002830.SZSE",
  "600979.SSE",
  "688037.SSE",
  "002594.SZSE",
  "300360.SZSE",
  "300581.SZSE",
  "301577.SZSE",
  "605259.SSE",
  "000421.SZSE",
  "002541.SZSE",
  "300315.SZSE",
  "002393.SZSE",
  "600761.SSE",
  "000025.SZSE",
  "300771.SZSE",
  "301193.SZSE",
  "002081.SZSE",
  "600956.SSE",
  "300635.SZSE",
  "688660.SSE",
  "300755.SZSE",
  "600882.SSE",
  "300605.SZSE",
  "688508.SSE",
  "000732.SZSE",
  "688687.SSE",
  "300381.SZSE",
  "603191.SSE",
  "600367.SSE",
  "301220.SZSE",
  "601788.SSE",
  "603698.SSE",
  "688377.SSE",
  "301329.SZSE",
  "300826.SZSE",
  "688759.SSE",
  "688518.SSE",
  "300407.SZSE",
  "300449.SZSE",
  "603668.SSE",
  "000950.SZSE",
  "000928.SZSE",
  "600589.SSE",
  "002005.SZSE",
  "002661.SZSE",
  "600983.SSE",
  "600012.SSE",
  "603711.SSE",
  "600288.SSE",
  "603577.SSE",
  "300942.SZSE",
  "603922.SSE",
  "301168.SZSE",
  "603585.SSE",
  "301322.SZSE",
  "688479.SSE",
  "603226.SSE",
  "301068.SZSE",
  "000059.SZSE",
  "601028.SSE",
  "002788.SZSE",
  "000055.SZSE",
  "301600.SZSE",
  "301006.SZSE",
  "603859.SSE",
  "301533.SZSE",
  "300766.SZSE",
  "300032.SZSE",
  "688093.SSE",
  "605086.SSE",
  "000797.SZSE",
  "600814.SSE",
  "000619.SZSE",
  "300307.SZSE",
  "002479.SZSE",
  "600660.SSE",
  "002903.SZSE",
  "002358.SZSE",
  "301479.SZSE",
  "300486.SZSE",
  "600393.SSE",
  "688382.SSE",
  "600597.SSE",
  "603618.SSE",
  "603605.SSE",
  "688045.SSE",
  "002736.SZSE",
  "688553.SSE",
  "300450.SZSE",
  "600683.SSE",
  "688677.SSE",
  "001965.SZSE",
  "301066.SZSE",
  "603477.SSE",
  "002298.SZSE",
  "002970.SZSE",
  "002184.SZSE",
  "601127.SSE",
  "688288.SSE",
  "301098.SZSE",
  "002299.SZSE",
  "002618.SZSE",
  "605289.SSE",
  "002120.SZSE",
  "002615.SZSE",
  "688626.SSE",
  "301348.SZSE",
  "605028.SSE",
  "300986.SZSE",
  "300779.SZSE",
  "600577.SSE",
  "600169.SSE",
  "688428.SSE",
  "601606.SSE",
  "688718.SSE",
  "300181.SZSE",
  "605168.SSE",
  "002905.SZSE",
  "601009.SSE",
  "688505.SSE",
  "603322.SSE",
  "300887.SZSE",
  "002292.SZSE",
  "688686.SSE",
  "600462.SSE",
  "000923.SZSE",
  "688390.SSE",
  "600565.SSE",
  "000567.SZSE",
  "601858.SSE",
  "000560.SZSE",
  "000938.SZSE",
  "600311.SSE",
  "002600.SZSE",
  "000061.SZSE",
  "600791.SSE",
  "600875.SSE",
  "000698.SZSE",
  "688099.SSE",
  "300465.SZSE",
  "601360.SSE",
  "301373.SZSE",
  "002722.SZSE",
  "002845.SZSE",
  "002524.SZSE",
  "002935.SZSE",
  "601618.SSE",
  "603429.SSE",
  "600084.SSE",
  "603569.SSE",
  "600530.SSE",
  "002825.SZSE",
  "300769.SZSE",
  "600662.SSE",
  "605266.SSE",
  "688070.SSE",
  "601021.SSE",
  "688392.SSE",
  "600626.SSE",
  "002625.SZSE",
  "600241.SSE",
  "688272.SSE",
  "688007.SSE",
  "603326.SSE",
  "600481.SSE",
  "300753.SZSE",
  "603696.SSE",
  "000628.SZSE",
  "688778.SSE",
  "688312.SSE",
  "000547.SZSE",
  "603356.SSE",
  "601216.SSE",
  "002296.SZSE",
  "002136.SZSE",
  "000953.SZSE",
  "300863.SZSE",
  "300261.SZSE",
  "688279.SSE",
  "300052.SZSE",
  "603124.SSE",
  "301332.SZSE",
  "301448.SZSE",
  "688607.SSE",
  "001359.SZSE",
  "300255.SZSE",
  "301236.SZSE",
  "688638.SSE",
  "000889.SZSE",
  "601226.SSE",
  "688538.SSE",
  "301568.SZSE",
  "300208.SZSE",
  "300512.SZSE",
  "002381.SZSE",
  "300278.SZSE",
  "688565.SSE",
  "603609.SSE",
  "600585.SSE",
  "300662.SZSE",
  "688389.SSE",
  "300441.SZSE",
  "603599.SSE",
  "002468.SZSE",
  "300396.SZSE",
  "688292.SSE",
  "300078.SZSE",
  "688567.SSE",
  "301187.SZSE",
  "301209.SZSE",
  "000498.SZSE",
  "603912.SSE",
  "300335.SZSE",
  "300099.SZSE",
  "688290.SSE",
  "600929.SSE",
  "600456.SSE",
  "603790.SSE",
  "300041.SZSE",
  "600570.SSE",
  "002354.SZSE",
  "601108.SSE",
  "605138.SSE",
  "002650.SZSE",
  "002222.SZSE",
  "002342.SZSE",
  "300207.SZSE",
  "000736.SZSE",
  "688180.SSE",
  "600845.SSE",
  "002884.SZSE",
  "001373.SZSE",
  "601077.SSE",
  "688788.SSE",
  "002550.SZSE",
  "002318.SZSE",
  "601158.SSE",
  "000961.SZSE",
  "000983.SZSE",
  "002588.SZSE",
  "600704.SSE",
  "002107.SZSE",
  "003035.SZSE",
  "002160.SZSE",
  "600083.SSE",
  "002194.SZSE",
  "300715.SZSE",
  "688019.SSE",
  "300804.SZSE",
  "300880.SZSE",
  "600160.SSE",
  "601991.SSE",
  "688581.SSE",
  "688297.SSE",
  "002135.SZSE",
  "300035.SZSE",
  "601298.SSE",
  "603303.SSE",
  "001213.SZSE",
  "603858.SSE",
  "300625.SZSE",
  "601390.SSE",
  "002002.SZSE",
  "600858.SSE",
  "600509.SSE",
  "002022.SZSE",
  "600859.SSE",
  "002946.SZSE",
  "301189.SZSE",
  "002281.SZSE",
  "603617.SSE",
  "002713.SZSE",
  "688496.SSE",
  "300003.SZSE",
  "603355.SSE",
  "002598.SZSE",
  "688039.SSE",
  "000815.SZSE",
  "605009.SSE",
  "002606.SZSE",
  "601218.SSE",
  "688102.SSE",
  "600556.SSE",
  "002802.SZSE",
  "002025.SZSE",
  "688028.SSE",
  "301203.SZSE",
  "300445.SZSE",
  "603078.SSE",
  "300919.SZSE",
  "301262.SZSE",
  "003031.SZSE",
  "301067.SZSE",
  "300130.SZSE",
  "002447.SZSE",
  "603110.SSE",
  "603026.SSE",
  "603288.SSE",
  "002572.SZSE",
  "688025.SSE",
  "301275.SZSE",
  "600372.SSE",
  "600240.SSE",
  "601965.SSE",
  "300739.SZSE",
  "603070.SSE",
  "300279.SZSE",
  "300689.SZSE",
  "600777.SSE",
  "000962.SZSE",
  "300723.SZSE",
  "603006.SSE",
  "002118.SZSE",
  "688077.SSE",
  "002733.SZSE",
  "688282.SSE",
  "002032.SZSE",
  "603257.SSE",
  "002383.SZSE",
  "600460.SSE",
  "002876.SZSE",
  "600479.SSE",
  "605088.SSE"
]

In [9]:
print(len(done))

3391


In [10]:
task_symbols = component_symbols  + ["866011.VTRI"]  # 目标股票
task_symbols = list(set(task_symbols) - set(done))
print(len(task_symbols))

2051


In [11]:
# ============================================================================
# Cell 6:
# ============================================================================

# 4.2 下载k线
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)


n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval2, adjust_type='post')
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

  0%|          | 0/2051 [00:00<?, ?it/s]

下载 603395.SSE ...


  0%|          | 1/2051 [00:02<1:16:10,  2.23s/it]

股票1
  ->  374 条K线
下载 002613.SZSE ...


  0%|          | 3/2051 [00:02<22:58,  1.49it/s]  

...

股票2048
  ->  1450 条K线
下载 300191.SZSE ...


100%|█████████▉| 2050/2051 [08:47<00:00,  5.22it/s]

股票2049
  ->  3262 条K线
下载 688510.SSE ...
股票2050
  ->  1329 条K线
下载 002477.SZSE ...


100%|██████████| 2051/2051 [08:47<00:00,  3.89it/s]

股票2051
  ->  1647 条K线
剩余0只未下载
